<a href="https://colab.research.google.com/github/icosahedron31/Walmart-Sales/blob/main/model_experiment_Prophet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install prophet mlflow dagshub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
from prophet import Prophet
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive

drive.mount('/content/drive')

dagshub.init(
    repo_owner='icosahedron31',
    repo_name='Walmart-Sales',
    mlflow=True
)

mlflow.set_experiment('Prophet_Training ')

Mounted at /content/drive


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=bf7d3d92-b23b-4bff-a5ca-3cf14a8ddca9&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=dcce71300cd7c5c66ce2d2e073f450d7cdacc7417d68a918e41cfb70f5a7b33c




Accessing as njvar23

Initialized MLflow to track repo "icosahedron31/Walmart-Sales"

Repository icosahedron31/Walmart-Sales initialized!

<Experiment: artifact_location='mlflow-artifacts:/408c625a68f04124a43ce7664f934f8e', creation_time=1783673999312, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1783673999312, lifecycle_stage='active', name='Prophet_Training ', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [ ]:
train_raw = pd.read_csv('/content/drive/MyDrive/walmart/train.csv')
features = pd.read_csv('/content/drive/MyDrive/walmart/features.csv')
train_raw['Date'] = pd.to_datetime(train_raw['Date'])
features['Date'] = pd.to_datetime(features['Date'])

# Split into train and val periods
train_period = train_raw[train_raw['Date'] <= pd.to_datetime('2012-04-01')]
val_period = train_raw[train_raw['Date'] > pd.to_datetime('2012-04-01')]

# Create Prophet format
nf_train = train_period[['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday']].copy()
nf_train['unique_id'] = nf_train['Store'].astype(str) + '_' + nf_train['Dept'].astype(str)
nf_train = nf_train.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})

nf_val = val_period[['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday']].copy()
nf_val['unique_id'] = nf_val['Store'].astype(str) + '_' + nf_val['Dept'].astype(str)
nf_val = nf_val.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})

# Filter short series
series_counts = nf_train.groupby('unique_id')['ds'].count()
valid_series = series_counts[series_counts >= 52].index
nf_train = nf_train[nf_train['unique_id'].isin(valid_series)].copy()
nf_val = nf_val[nf_val['unique_id'].isin(valid_series)].copy()

horizon = nf_val['ds'].nunique()

print(f"Train: {nf_train['ds'].min()} → {nf_train['ds'].max()}")
print(f"Val: {nf_val['ds'].min()} → {nf_val['ds'].max()}")
print(f"Unique series: {nf_train['unique_id'].nunique()}")
print(f"Horizon: {horizon} weeks")

Train: 2010-02-05 00:00:00 → 2012-03-30 00:00:00
Val: 2012-04-06 00:00:00 → 2012-10-26 00:00:00
Unique series: 2960
Horizon: 30 weeks


In [ ]:
# Define holidays for Prophet
holidays = pd.DataFrame({
    'holiday': [
        'superbowl', 'superbowl', 'superbowl', 'superbowl',
        'laborday', 'laborday', 'laborday', 'laborday',
        'thanksgiving', 'thanksgiving', 'thanksgiving',
        'christmas', 'christmas', 'christmas'
    ],
    'ds': pd.to_datetime([
        '2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08',
        '2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06',
        '2010-11-26', '2011-11-25', '2012-11-23',
        '2010-12-31', '2011-12-30', '2012-12-28'
    ]),
    'lower_window': -1,   # 1 week before holiday
    'upper_window': 1     # 1 week after holiday
})

print(holidays)

         holiday         ds  lower_window  upper_window
0      superbowl 2010-02-12            -1             1
1      superbowl 2011-02-11            -1             1
2      superbowl 2012-02-10            -1             1
3      superbowl 2013-02-08            -1             1
4       laborday 2010-09-10            -1             1
5       laborday 2011-09-09            -1             1
6       laborday 2012-09-07            -1             1
7       laborday 2013-09-06            -1             1
8   thanksgiving 2010-11-26            -1             1
9   thanksgiving 2011-11-25            -1             1
10  thanksgiving 2012-11-23            -1             1
11     christmas 2010-12-31            -1             1
12     christmas 2011-12-30            -1             1
13     christmas 2012-12-28            -1             1


In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [ ]:
with mlflow.start_run(run_name='Prophet_Cleaning'):
    mlflow.log_param('train_rows', len(nf_train))
    mlflow.log_param('val_rows', len(nf_val))
    mlflow.log_param('unique_series', nf_train['unique_id'].nunique())
    mlflow.log_param('horizon', horizon)
    mlflow.log_param('cutoff_date', '2012-04-01')
    mlflow.log_param('min_series_length', 52)
    print("Cleaning run logged")

Cleaning run logged
🏃 View run Prophet_Cleaning at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/20371bb47ae14049a9510a1105b4f743
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5


In [ ]:
def fit_predict_prophet(train_series, val_dates, model_params, holidays=None):
    """
    Fits Prophet on one series and returns predictions for val dates.

    Args:
        train_series: DataFrame with ds and y columns
        val_dates: DataFrame with ds column
        model_params: dict of Prophet parameters
        holidays: DataFrame of holidays (optional)

    Returns:
        DataFrame with ds and yhat columns
    """
    m = Prophet(
        holidays=holidays,
        **model_params
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m.fit(train_series[['ds', 'y']])

    forecast = m.predict(val_dates[['ds']])
    return forecast[['ds', 'yhat']]


def run_prophet_experiment(nf_train, nf_val, model_params, holidays=None):
    """
    Runs Prophet on all series and returns merged predictions.
    """
    all_forecasts = []
    series_ids = nf_train['unique_id'].unique()

    for uid in tqdm(series_ids, desc="Fitting Prophet"):
        train_series = nf_train[nf_train['unique_id'] == uid].copy()
        val_dates = nf_val[nf_val['unique_id'] == uid][['ds']].copy()

        if len(val_dates) == 0 or len(train_series) < 10:
            continue

        try:
            forecast = fit_predict_prophet(
                train_series, val_dates, model_params, holidays
            )
            forecast['unique_id'] = uid
            all_forecasts.append(forecast)
        except Exception as e:
            continue

    forecasts_df = pd.concat(all_forecasts, ignore_index=True)

    val_merged = nf_val.merge(
        forecasts_df, on=['unique_id', 'ds'], how='inner'
    )

    return val_merged

In [ ]:
with mlflow.start_run(run_name='Prophet_Baseline_Additive'):

    model_params = {
        'yearly_seasonality': True,
        'weekly_seasonality': False,
        'daily_seasonality': False,
        'seasonality_mode': 'additive'
    }

    val_merged = run_prophet_experiment(nf_train, nf_val, model_params)

    score = wmae(
        val_merged['y'].values,
        val_merged['yhat'].values,
        val_merged['IsHoliday'].fillna(False).values
    )

    mlflow.log_params(model_params)
    mlflow.log_param('holidays', False)
    mlflow.log_metric('wmae', score)
    mlflow.log_metric('series_predicted', val_merged['unique_id'].nunique())

    mlflow.log_metric('wmae', score)
    mlflow.log_metric('series_predicted', val_merged['unique_id'].nunique())
    mlflow.log_metric('coverage_pct',
        val_merged['unique_id'].nunique() / nf_val['unique_id'].nunique() * 100
    )
    mlflow.log_metric('median_series_mae',
        val_merged.groupby('unique_id').apply(
            lambda x: np.mean(np.abs(x['y'] - x['yhat']))
        ).median()
    )

    print(f"Baseline Additive WMAE: {score:.2f}")

Fitting Prophet: 100%|██████████| 2960/2960 [05:21<00:00,  9.19it/s]


Baseline Additive WMAE: 1702.19
🏃 View run Prophet_Baseline_Additive at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/b3828c68b04d49a08b403c07f0ac6779
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5


In [ ]:
with mlflow.start_run(run_name='Prophet_Multiplicative'):

    model_params = {
        'yearly_seasonality': True,
        'weekly_seasonality': False,
        'daily_seasonality': False,
        'seasonality_mode': 'multiplicative'
    }

    val_merged = run_prophet_experiment(nf_train, nf_val, model_params)

    score = wmae(
        val_merged['y'].values,
        val_merged['yhat'].values,
        val_merged['IsHoliday'].fillna(False).values
    )

    # Parameters
    mlflow.log_params(model_params)
    mlflow.log_param('holidays', False)
    mlflow.log_param('n_series', nf_train['unique_id'].nunique())
    mlflow.log_param('horizon_weeks', horizon)

    # Metrics
    mlflow.log_metric('wmae', score)
    mlflow.log_metric('series_predicted', val_merged['unique_id'].nunique())
    mlflow.log_metric('coverage_pct',
        val_merged['unique_id'].nunique() / nf_val['unique_id'].nunique() * 100
    )
    mlflow.log_metric('median_series_mae',
        val_merged.groupby('unique_id').apply(
            lambda x: np.mean(np.abs(x['y'] - x['yhat']))
        ).median()
    )

    print(f"Multiplicative WMAE: {score:.2f}")

Fitting Prophet: 100%|██████████| 2960/2960 [05:29<00:00,  8.97it/s]


Multiplicative WMAE: 1689.76
🏃 View run Prophet_Multiplicative at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/e932a12b0f634c99800aaab0e6225077
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5


In [ ]:
with mlflow.start_run(run_name='Prophet_Multiplicative_Holidays'):

    model_params = {
        'yearly_seasonality': True,
        'weekly_seasonality': False,
        'daily_seasonality': False,
        'seasonality_mode': 'multiplicative',
        'changepoint_prior_scale': 0.05,
        'seasonality_prior_scale': 10
    }

    val_merged = run_prophet_experiment(
        nf_train, nf_val, model_params, holidays=holidays
    )

    score = wmae(
        val_merged['y'].values,
        val_merged['yhat'].values,
        val_merged['IsHoliday'].fillna(False).values
    )

    mlflow.log_params(model_params)
    mlflow.log_param('holidays', True)
    mlflow.log_param('holiday_window', '-1 to +1')
    mlflow.log_param('n_series', nf_train['unique_id'].nunique())
    mlflow.log_param('horizon_weeks', horizon)

    mlflow.log_metric('wmae', score)
    mlflow.log_metric('series_predicted', val_merged['unique_id'].nunique())
    mlflow.log_metric('coverage_pct',
        val_merged['unique_id'].nunique() / nf_val['unique_id'].nunique() * 100
    )
    mlflow.log_metric('median_series_mae',
        val_merged.groupby('unique_id').apply(
            lambda x: np.mean(np.abs(x['y'] - x['yhat']))
        ).median()
    )

    print(f"Multiplicative + Holidays WMAE: {score:.2f}")

Fitting Prophet: 100%|██████████| 2960/2960 [06:58<00:00,  7.08it/s]


Multiplicative + Holidays WMAE: 1665.05
🏃 View run Prophet_Multiplicative_Holidays at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/c3ad146872a14579ad52b54f341ac867
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5


In [ ]:
for fourier_order in [5, 10, 20]:
    with mlflow.start_run(run_name=f'Prophet_Fourier_{fourier_order}'):

        model_params = {
            'yearly_seasonality': fourier_order,
            'weekly_seasonality': False,
            'daily_seasonality': False,
            'seasonality_mode': 'multiplicative',
            'changepoint_prior_scale': 0.05,
            'seasonality_prior_scale': 10
        }

        val_merged = run_prophet_experiment(
            nf_train, nf_val, model_params, holidays=holidays
        )

        score = wmae(
            val_merged['y'].values,
            val_merged['yhat'].values,
            val_merged['IsHoliday'].fillna(False).values
        )

        mlflow.log_params(model_params)
        mlflow.log_param('holidays', True)
        mlflow.log_param('fourier_order', fourier_order)
        mlflow.log_param('n_series', nf_train['unique_id'].nunique())
        mlflow.log_param('horizon_weeks', horizon)

        mlflow.log_metric('wmae', score)
        mlflow.log_metric('series_predicted', val_merged['unique_id'].nunique())
        mlflow.log_metric('coverage_pct',
            val_merged['unique_id'].nunique() / nf_val['unique_id'].nunique() * 100
        )
        mlflow.log_metric('median_series_mae',
            val_merged.groupby('unique_id').apply(
                lambda x: np.mean(np.abs(x['y'] - x['yhat']))
            ).median()
        )

        print(f"Fourier order={fourier_order} WMAE: {score:.2f}")

Fitting Prophet: 100%|██████████| 2960/2960 [06:28<00:00,  7.61it/s]


Fourier order=5 WMAE: 1730.27
🏃 View run Prophet_Fourier_5 at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/9c33c9a20fab40bfbcb9e7e4bbdcdeab
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5


Fitting Prophet: 100%|██████████| 2960/2960 [07:04<00:00,  6.98it/s]


Fourier order=10 WMAE: 1665.05
🏃 View run Prophet_Fourier_10 at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/9234561bc0374420a5e31fc8aaca575a
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5


Fitting Prophet: 100%|██████████| 2960/2960 [08:38<00:00,  5.71it/s]


Fourier order=20 WMAE: 1919.11
🏃 View run Prophet_Fourier_20 at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/91bb7bfbea0e4d37a8592557d8b65b6e
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5


In [ ]:
for cp_scale in [0.01, 0.05, 0.1, 0.5]:
    with mlflow.start_run(run_name=f'Prophet_CP_{cp_scale}'):

        model_params = {
            'yearly_seasonality': 10,
            'weekly_seasonality': False,
            'daily_seasonality': False,
            'seasonality_mode': 'multiplicative',
            'changepoint_prior_scale': cp_scale,
            'seasonality_prior_scale': 10
        }

        val_merged = run_prophet_experiment(
            nf_train, nf_val, model_params, holidays=holidays
        )

        score = wmae(
            val_merged['y'].values,
            val_merged['yhat'].values,
            val_merged['IsHoliday'].fillna(False).values
        )

        mlflow.log_params(model_params)
        mlflow.log_param('holidays', True)
        mlflow.log_param('n_series', nf_train['unique_id'].nunique())
        mlflow.log_param('horizon_weeks', horizon)

        mlflow.log_metric('wmae', score)
        mlflow.log_metric('series_predicted', val_merged['unique_id'].nunique())
        mlflow.log_metric('coverage_pct',
            val_merged['unique_id'].nunique() / nf_val['unique_id'].nunique() * 100
        )
        mlflow.log_metric('median_series_mae',
            val_merged.groupby('unique_id').apply(
                lambda x: np.mean(np.abs(x['y'] - x['yhat']))
            ).median()
        )

        print(f"CP scale={cp_scale} WMAE: {score:.2f}")

Fitting Prophet: 100%|██████████| 2960/2960 [07:04<00:00,  6.98it/s]


CP scale=0.01 WMAE: 1938.31
🏃 View run Prophet_CP_0.01 at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/14631a13ab7842bd8c91b6c50348d9b1
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5


Fitting Prophet: 100%|██████████| 2960/2960 [07:08<00:00,  6.90it/s]


CP scale=0.05 WMAE: 1665.05
🏃 View run Prophet_CP_0.05 at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/2322a7be2e814eeab94a0c32376fc2c2
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5


Fitting Prophet: 100%|██████████| 2960/2960 [07:25<00:00,  6.65it/s]


CP scale=0.1 WMAE: 1737.22
🏃 View run Prophet_CP_0.1 at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/2c5c0d92d3594125b89cb0030f40d44b
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5


Fitting Prophet: 100%|██████████| 2960/2960 [08:29<00:00,  5.81it/s]


CP scale=0.5 WMAE: 1994.06
🏃 View run Prophet_CP_0.5 at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/c6a2d42046344b70900b0c1f47022d40
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5


In [ ]:
with mlflow.start_run(run_name='Prophet_Pipeline_Final'):

    best_params = {
        'yearly_seasonality': True,
        'weekly_seasonality': False,
        'daily_seasonality': False,
        'seasonality_mode': 'multiplicative',
        'changepoint_prior_scale': 0.05,
        'seasonality_prior_scale': 10
    }

    all_models = {}
    all_forecasts = []

    for uid in tqdm(nf_train['unique_id'].unique(), desc="Training final Prophet"):
        train_series = nf_train[nf_train['unique_id'] == uid].copy()
        val_dates = nf_val[nf_val['unique_id'] == uid][['ds']].copy()

        if len(val_dates) == 0 or len(train_series) < 10:
            continue

        try:
            m = Prophet(holidays=holidays, **best_params)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                m.fit(train_series[['ds', 'y']])
            all_models[uid] = m
            forecast = m.predict(val_dates)
            forecast['unique_id'] = uid
            all_forecasts.append(forecast[['unique_id', 'ds', 'yhat']])
        except Exception:
            continue

    forecasts_df = pd.concat(all_forecasts, ignore_index=True)
    val_merged = nf_val.merge(forecasts_df, on=['unique_id', 'ds'], how='inner')
    val_merged['yhat'] = val_merged['yhat'].clip(lower=0)

    score = wmae(
        val_merged['y'].values,
        val_merged['yhat'].values,
        val_merged['IsHoliday'].fillna(False).values
    )

    mlflow.log_params(best_params)
    mlflow.log_param('holidays', True)
    mlflow.log_metric('wmae', score)
    mlflow.log_metric('series_trained', len(all_models))

    # Save as pyfunc — only one model saved
    class ProphetPyfunc(mlflow.pyfunc.PythonModel):
        def __init__(self, models_dict):
            self.models_dict = models_dict

        def predict(self, context, model_input):
            all_forecasts = []
            for uid in model_input['unique_id'].unique():
                if uid not in self.models_dict:
                    continue
                dates = model_input[model_input['unique_id'] == uid][['ds']].copy()
                dates['ds'] = pd.to_datetime(dates['ds'])
                m = self.models_dict[uid]
                forecast = m.predict(dates)
                forecast['unique_id'] = uid
                all_forecasts.append(forecast[['unique_id', 'ds', 'yhat']])
            result = pd.concat(all_forecasts, ignore_index=True)
            result['yhat'] = result['yhat'].clip(lower=0)
            return result

    prophet_model = ProphetPyfunc(all_models)

    mlflow.pyfunc.log_model(
        artifact_path='prophet_pipeline',
        python_model=prophet_model,
        registered_model_name='Prophet_Best'
    )

    print(f"Final Prophet WMAE: {score:.2f}")
    print(f"Models trained: {len(all_models)}")
    print("Registered as Prophet_Best")

Training final Prophet: 100%|██████████| 2960/2960 [11:04<00:00,  4.45it/s]
/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/07/11 10:18:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 10:18:09 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Registered model 'Prophet_Best' already exists. Creating a new version of this model...
2026/

Final Prophet WMAE: 1653.38
Models trained: 2952
Registered as Prophet_Best
🏃 View run Prophet_Pipeline_Final at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5/runs/cc40575870c8430a8b97927b1c7f906e
🧪 View experiment at: https://dagshub.com/icosahedron31/Walmart-Sales.mlflow/#/experiments/5
